In [4]:
!pip install triton-windows

   ---------------------------------------- 0.0/49.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/49.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/49.6 MB ? eta -:--:--
    --------------------------------------- 0.8/49.6 MB 2.0 MB/s eta 0:00:25
    --------------------------------------- 1.0/49.6 MB 1.9 MB/s eta 0:00:27
   - -------------------------------------- 1.6/49.6 MB 2.1 MB/s eta 0:00:23
   - -------------------------------------- 1.8/49.6 MB 2.0 MB/s eta 0:00:25
   - -------------------------------------- 1.8/49.6 MB 2.0 MB/s eta 0:00:25
   - -------------------------------------- 2.4/49.6 MB 1.6 MB/s eta 0:00:30
   -- ------------------------------------- 2.9/49.6 MB 1.8 MB/s eta 0:00:27
   -- ------------------------------------- 3.4/49.6 MB 1.8 MB/s eta 0:00:26
   -- ------------------------------------- 3.7/49.6 MB 1.8 MB/s eta 0:00:26
   --- ------------------------------------ 3.9/49.6 MB 1.8 MB/s eta 0:00:26
   --- -------------

In [5]:
import torch
import triton
import triton.language as tl

In [6]:
torch.cuda.is_available()
torch.cuda.get_device_name(0)

'NVIDIA GeForce GTX 1650 Ti'

In [7]:

# 1. THE KERNEL (This code executes directly on the GPU cores)
@triton.jit
def add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # Figure out which parallel instance (program) this execution thread belongs to
    pid = tl.program_id(axis=0)
    
    # Calculate the memory offset boundaries for this specific block
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    
    # Create a boundary guard mask so we don't read out-of-bounds memory
    mask = offsets < n_elements
    
    # Load data from slow Global Memory (DRAM) into ultra-fast on-chip registers
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    
    # Perform the execution in parallel
    output = x + y
    
    # Write the results back out to Global Memory
    tl.store(output_ptr + offsets, output, mask=mask)

# 2. THE HOST WRAPPER (This coordinates execution from the CPU)
def triton_add(x: torch.Tensor, y: torch.Tensor):
    output = torch.empty_like(x)
    n_elements = output.numel()
    
    # Define the Grid: how many parallel blocks do we need to launch?
    # cdiv divides and rounds up (e.g., if elements don't perfectly fit into blocks)
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    
    # Launch the kernel on the GPU
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output

# 3. VERIFICATION CODE
size = 100_000
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')

# Run native PyTorch operation
output_torch = x + y

# Run your custom GPU kernel
output_triton = triton_add(x, y)

# Compare the results
max_diff = torch.max(torch.abs(output_torch - output_triton)).item()
print(f"Kernel Execution Successful!")
print(f"Max numerical difference from PyTorch: {max_diff}")

Kernel Execution Successful!
Max numerical difference from PyTorch: 0.0
